In [ ]:
import numpy as np
tour = [1, 2, 3, 4, 5]

In [ ]:
tour_shifted = np.roll(tour, -1)
tour_shifted

In [ ]:
def tour_distance(tour, dist_matrix):
    tour_shifted = np.roll(tour, -1)
    return np.sum(np.asarray(dist_matrix)[tour, tour_shifted])

In [ ]:
import numpy as np

tour = np.array([0, 1, 2, 3, 4, 5, 6])  # visits all 7 cities then returns to 0
dist_matrix = np.array([
    [0, 10, 20, 25, 31, 40, 42],
    [10, 0, 15, 22, 27, 33, 35],
    [20, 15, 0, 18, 24, 30, 32],
    [25, 22, 18, 0, 16, 26, 28],
    [31, 27, 24, 16, 0, 21, 23],
    [40, 33, 30, 26, 21, 0, 19],
    [42, 35, 32, 28, 23, 19, 0],
])

total = tour_distance(tour, dist_matrix)
print(total)

## Testing 2-opt

In [ ]:
tour

In [ ]:
def two_opt(tour, i, j):
    print("tour:", tour, "i:", i, "j:", j)
    print("tour segment to be reversed:", tour[i:j+1])
    new_tour = np.concatenate((tour[:i], tour[i:j+1][::-1], tour[j+1:]))
    return new_tour

In [ ]:
def _two_opt_first_improvement(tour: np.ndarray) -> tuple[np.ndarray, bool]:
    current_distance = 10
    length = len(tour) - 1 if tour[0] == tour[-1] else len(tour)
    for i in range(1, length - 1):
        for j in range(i + 1, length):
            if j - i == 1:
                continue
            candidate = two_opt(tour, i, j)
            print("New tour:", candidate)
            candidate_distance = 11
            if candidate_distance < current_distance:
                return candidate, True
    return tour, False

In [ ]:
_two_opt_first_improvement(tour)

In [ ]:
is_closed = tour[0] == tour[-1]
length = len(tour) - 1 if is_closed else len(tour)

def edge_cost(u: int, v: int) -> float:
    lower, upper = (u, v) if u <= v else (v, u)
    return dist_matrix[lower, upper]

core = tour[:-1] if is_closed else tour

In [ ]:
for i in range(1, length - 1):
    for j in range(i + 1, length):
        if j - i == 1:
            continue
        
        print("i:", i, "j:", j)
        a, b = core[i - 1], core[i]
        c, d = core[j], core[(j + 1) % length]
        print("a, b, c, d:", a, b, c, d)

        removed = edge_cost(a, b) + edge_cost(c, d)
        added = edge_cost(a, c) + edge_cost(b, d)
        delta = added - removed
        

In [ ]:
def double_bridge_move(tour: np.ndarray, a: int, b: int, c: int, d: int) -> np.ndarray:
    """Apply a double-bridge move defined by four cut indices."""
    if len(tour) < 6:
        return tour.copy()

    is_closed = tour[0] == tour[-1]
    core = tour[:-1] if is_closed else tour
    n = len(core)
    print("Core:", core)
    if n < 6:
        return tour.copy()

    segment_1 = core[:a]
    segment_2 = core[a:b]
    segment_3 = core[b:c]
    segment_4 = core[c:d]
    segment_5 = core[d:]
    print("Segments:", segment_1, segment_2, segment_3, segment_4, segment_5)

    new_core = np.concatenate((segment_1, segment_3, segment_2, segment_4, segment_5))
    if is_closed:
        new_core = np.concatenate((new_core, [new_core[0]]))
    return new_core

### One insertion

In [ ]:
def one_insertion(tour: np.ndarray, i: int, j: int) -> np.ndarray:
    """Remove the vertex at position i and insert it before position j."""
    if i == j:
        return tour.copy()

    is_closed = tour[0] == tour[-1]
    core = tour[:-1] if is_closed else tour
    new_core = core.copy()
    city = new_core[i]
    new_core = np.delete(new_core, i)
    print("tour:", tour, "i:", i, "j:", j, "new_core before insertion:", new_core)
    if j > i:
        j -= 1
    new_core = np.insert(new_core, j, city)
    if is_closed:
        return np.concatenate((new_core, [new_core[0]]))
    print("new_core after insertion:", new_core)
    return new_core

In [ ]:
def _one_insertion_first_improvement(
    tour: np.ndarray, current_distance: float
) -> tuple[np.ndarray, bool]:
    length = len(tour) - 1 if tour[0] == tour[-1] else len(tour)
    if length < 3:
        return tour, False, current_distance

    for i in range(1, length):
        for j in range(1, length + 1):
            if i == j or j == i + 1:
                continue
            candidate = one_insertion(tour, i, j)
            candidate_distance = tour_distance(candidate, dist_matrix)
            if candidate_distance < current_distance:
                print("Improvement found:", candidate_distance, "<", current_distance)
                    
    print("No improvement found")   
    return tour, False

In [ ]:
tour_distance(tour, dist_matrix)

In [ ]:
_one_insertion_first_improvement(tour, 141)

In [ ]:
def _one_insertion_first_test(
     tour: np.ndarray, current_distance: float
) -> tuple[np.ndarray, bool, float]:
    """Return the first improving 1-insertion move using edge deltas."""

    is_closed = tour[0] == tour[-1]
    core = tour[:-1] if is_closed else tour
    length = len(core)
    if length < 3:
        return tour, False, current_distance

    def edge_cost(u: int, v: int) -> float:
        lower, upper = (u, v) if u <= v else (v, u)
        return dist_matrix[lower, upper]

    for i in range(1, length):
        city = core[i]
        prev_i = core[i - 1] if i > 0 else (core[-1] if is_closed else None)
        next_i = core[(i + 1) % length] if (is_closed or i + 1 < length) else None

        removal_delta = 0.0
        if prev_i is not None:
            removal_delta -= edge_cost(prev_i, city)
        if next_i is not None:
            removal_delta -= edge_cost(city, next_i)
            if prev_i is not None:
                removal_delta += edge_cost(prev_i, next_i)

        core_removed = np.delete(core, i)
        reduced_len = len(core_removed)
        if reduced_len == 0:
            continue

        for j in range(1, length + 1):
            if i == j or j == i + 1:
                continue

            insert_idx = j
            if insert_idx > i:
                insert_idx -= 1

            if insert_idx < 0 or insert_idx > reduced_len:
                continue

            if is_closed:
                prev_new = core_removed[(insert_idx - 1) % reduced_len]
                next_new = core_removed[insert_idx % reduced_len]
            else:
                prev_new = core_removed[insert_idx - 1] if insert_idx > 0 else None
                next_new = core_removed[insert_idx] if insert_idx < reduced_len else None

            insertion_delta = 0.0
            if prev_new is not None and next_new is not None:
                insertion_delta -= edge_cost(prev_new, next_new)
            if prev_new is not None:
                insertion_delta += edge_cost(prev_new, city)
            if next_new is not None:
                insertion_delta += edge_cost(city, next_new)

            delta = removal_delta + insertion_delta
            print("i:", i, "j:", j, "delta:", delta)
            if delta < 0:
                new_distance = current_distance + delta
                candidate = one_insertion(tour, i, j)
                print("Improvement found:", new_distance, "<", current_distance)
                return candidate, True, new_distance

    return tour, False, current_distance

In [ ]:
_one_insertion_first_test(tour, 141)

In [ ]:
def _double_bridge_first_improvement(tour: np.ndarray) -> tuple[np.ndarray, bool]:
    current_distance = 10
    is_closed = tour[0] == tour[-1]
    length = len(tour) - 1 if is_closed else len(tour)
    if length < 6:
        return tour, False

    for a in range(1, length - 3):
        for b in range(a + 1, length - 2):
            for c in range(b + 1, length - 1):
                for d in range(c + 1, length):
                    print("Current tour:", tour, "Cuts:", a, b, c, d)
                    candidate = double_bridge_move(tour, a, b, c, d)
                    print("New tour:", candidate)
                    candidate_distance = 11
                    if candidate_distance < current_distance:
                        return candidate, True
    return tour, False

In [ ]:
_double_bridge_first_improvement(tour)

In [ ]:
def _reward_matrix(distance_matrix: np.ndarray) -> np.ndarray:
	"""Build the reward matrix r(s,a) = M_i / d_ij as described in the paper."""
	with np.errstate(divide="ignore"):
		mean_dist = distance_matrix.mean(axis=1)
		print(mean_dist)
		rewards = np.divide(mean_dist[:, None], distance_matrix, where=distance_matrix > 0)
	rewards[np.isinf(rewards)] = 0.0
	rewards[np.isnan(rewards)] = 0.0
	return rewards




### Testing q-learning initial solution

In [ ]:
import sys
from pathlib import Path
import numpy as np

repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [ ]:
src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
src_dir

from pathlib import Path
import numpy as np

from src.structures.graph import Graph
from src.solver.initial_solution import q_learning_tour, QLearningConfig, nearest_neighbour_tour

In [ ]:
def _reward_matrix(distance_matrix: np.ndarray) -> np.ndarray:
    """Build the reward matrix r(s,a) = M_i / d_ij as described in the paper."""
    with np.errstate(divide="ignore"):
        mean_dist = distance_matrix.mean(axis=1)
        print("Mean distance:", mean_dist)
        rewards = np.divide(
            distance_matrix, mean_dist[:, None], where=mean_dist[:, None] > 0
        )
    rewards[np.isinf(rewards)] = 0.0
    rewards[np.isnan(rewards)] = 0.0
    return rewards


In [ ]:
def _reward_matrix(distance_matrix: np.ndarray) -> np.ndarray:
    """Build the reward matrix r(s,a) = M_i / d_ij as described in the paper."""
    with np.errstate(divide="ignore"):
        mean_dist = distance_matrix.mean(axis=1)
        rewards = np.divide(
            mean_dist[:, None], distance_matrix, where=distance_matrix > 0
        )
    rewards[np.isinf(rewards)] = 0.0
    rewards[np.isnan(rewards)] = 0.0
    return rewards

In [ ]:
instance_path = Path("/Users/karinaassiniandreatta/Documents/01 phd/codes/tsp_rl_metaheuristic/instances/tsplib/pcb442.tsp")
graph = Graph.from_tsplib(instance_path)
distance_full = np.asarray(graph.build_adj_matrix_full(), dtype=float)
distance_matrix = np.asarray(graph._build_adj_matrix(), dtype=float)

#### Reward matrix test

In [ ]:
import numpy as np

sample_dist = np.array([
    [0.0, 2.0, 4.0],
    [2.0, 0.0, 6.0],
    [4.0, 6.0, 0.0],
])

sample_rewards = _reward_matrix(sample_dist)
sample_rewards

In [ ]:
2/ 2.66666667

In [ ]:
distance_full

In [ ]:
def tour_distance(tour, dist_matrix):
    tour = np.asarray(tour, dtype=int)
    tour_shifted = np.roll(tour, -1)
    lower = np.minimum(tour, tour_shifted)
    upper = np.maximum(tour, tour_shifted)
    return float(np.sum(dist_matrix[lower, upper]))

In [ ]:
# Select the best action based on the Q-values -> epsilon-greedy
# _epsilon_greedy_action

# Toy Q-table: rows = states, columns = actions
q_table = np.array([
    [0.5, 0.9, 0.8, 0.1, 0.2],
    [0.6, 0.2, 0.4, 0.7, 0.3],
    [0.3, 0.5, 0.6, 0.2, 0.4],
    [0.4, 0.1, 0.3, 0.9, 0.5],
    [0.2, 0.8, 0.7, 0.5, 0.6],
])

available_actions = [0, 1, 2, 3, 4]
state = 0
q_values = q_table[state]
print(q_values)
best_action = max(available_actions, key=lambda action: q_values[action])
best_action

In [ ]:
next_state = 1
unvisited = {2, 3, 4}
if unvisited:
    print(q_table[next_state, list(unvisited)])
    max_future = q_table[next_state, list(unvisited)].max() if unvisited else 0.0
else:
    max_future = 0.0
max_future

# nesse caso, o maximo futuro seria do 1 para o 3, que é 0.7

### Solution test

In [ ]:
instance_path = Path("/Users/karinaassiniandreatta/Documents/01 phd/codes/tsp_rl_metaheuristic/instances/tsplib/ch150.tsp")
graph = Graph.from_tsplib(instance_path)
distance_full = np.asarray(graph.build_adj_matrix_full(), dtype=float)
distance_matrix = np.asarray(graph._build_adj_matrix(), dtype=float)

In [ ]:

cfg = QLearningConfig(
    episodes=2000,
    alpha=0.3,
    gamma=0.8,
    epsilon=0.6,
    epsilon_decay=0.998,
    epsilon_min=0.1,
    monitor_interval=100,
    epsilon_reset_interval=500,
    epsilon_reset_value=0.35,
    #cache_dir=Path("/Users/karinaassiniandreatta/Documents/01 phd/codes/tsp_rl_metaheuristic/data/q_learning_cache"),
)
tour_q, distance_matrix = q_learning_tour(graph, start=0, cfg=cfg)

#When Δ_frob and Δ_max drop near zero yet your tour costs stay high, the policy is no longer changing—training stalled


In [ ]:
tour_q

In [ ]:
tour_distance(tour_q, distance_full)

In [ ]:
tour = nearest_neighbour_tour(graph)
tour

In [ ]:
tour_distance(tour, distance_matrix)

# Testing my vns_solver with q_learning

In [ ]:
# first step 

def _initialise_rl_table(rl_operator_index, operator_names: list[str]) -> None:
    if (
        rl_operator_index is None
        or set(rl_operator_index) != set(operator_names)
        or len(rl_operator_index) != len(operator_names)
    ):
        rl_operator_index = {
            name: idx for idx, name in enumerate(operator_names)
        }
        size = len(operator_names)
        rl_q_table = np.zeros((size, size), dtype=float)
    elif rl_q_table is None or rl_q_table.shape[0] != len(operator_names):
        size = len(operator_names)
        rl_q_table = np.zeros((size, size), dtype=float)
    print(
        "Initialized RL Q-table with operators: %s", rl_q_table
    )
    return rl_q_table

In [ ]:
operator_names = ["2-opt", "1-insertion", "double-bridge"]  
rl_q_table = _initialise_rl_table(None, operator_names)

In [ ]:
import random
def select_search(state_idx: int | None, available: list[int], rl_q_table = None, rl_epsilon = 0.6) -> int:
    if not available:
        raise ValueError("No local-search operators available for selection.")
    if rl_q_table is None or random.random() < rl_epsilon:
        print("Exploration: selecting random operator")
        return random.choice(available)

    if state_idx is None:
        preferences = rl_q_table.max(axis=0)
    else:
        preferences = rl_q_table[state_idx]

    best_idx = max(available, key=lambda idx: preferences[idx])
    return best_idx

In [ ]:
best_idx = select_search(None, [0, 1, 2], rl_q_table, rl_epsilon=0.6)
best_idx

In [ ]:
def positive_reward(previous_best: float, candidate_distance: float) -> float:
    if not np.isfinite(previous_best):
        print("Previous best is not finite, returning reward 1.0")
        return 1.0
    delta = previous_best - candidate_distance
    if delta <= 0:
        print("No improvement, returning reward 0.0")
        return 0.0
    return delta / max(previous_best, 1e-9)

In [ ]:
positive_reward(150.0, 140.0)

## Test local search methods

In [1]:
import numpy as np
import random

In [3]:
def tour_distance(tour, dist_matrix):
    tour = np.asarray(tour, dtype=int)
    tour_shifted = np.roll(tour, -1)
    lower = np.minimum(tour, tour_shifted)
    upper = np.maximum(tour, tour_shifted)
    return float(np.sum(dist_matrix[lower, upper]))

In [4]:
import time

tour = np.array([0, 1, 2, 3, 4, 5, 0])
dist_matrix = np.array([
    [0.0, 10.0, 20.0, 25.0, 31.0, 40.0, 42.0],
    [10.0, 0.0, 15.0, 22.0, 27.0, 33.0, 35.0],
    [20.0, 15.0, 0.0, 18.0, 24.0, 30.0, 32.0],
    [25.0, 22.0, 18.0, 0.0, 16.0, 26.0, 28.0],
    [31.0, 27.0, 24.0, 16.0, 0.0, 21.0, 23.0],
    [40.0, 33.0, 30.0, 26.0, 21.0, 0.0, 19.0],
    [42.0, 35.0, 32.0, 28.0, 23.0, 19.0, 0.0],
])

current_distance = tour_distance(tour, dist_matrix)

In [ ]:
def one_insertion(tour: np.ndarray, i: int, j: int) -> np.ndarray:
    """Remove the vertex at position i and insert it before position j."""
    if i == j:
        return tour.copy()

    is_closed = tour[0] == tour[-1]
    core = tour[:-1] if is_closed else tour
    new_core = core.copy()
    city = new_core[i]
    new_core = np.delete(new_core, i)
    if j > i:
        j -= 1
    new_core = np.insert(new_core, j, city)
    if is_closed:
        return np.concatenate((new_core, [new_core[0]]))
    return new_core




In [ ]:

def one_insertion_first_improvement(
    tour: np.ndarray,
    current_distance: float,
    distance_matrix: np.ndarray,
) -> tuple[np.ndarray, bool, float]:
    """Standalone 1-insertion first improvement search."""
    length = len(tour) - 1 if tour[0] == tour[-1] else len(tour)
    if length < 3:
        return tour.copy(), False, current_distance

    for i in range(1, length):
        for j in range(1, length + 1):
            if i == j or j == i + 1:
                continue
            candidate = one_insertion(tour, i, j)
            candidate_distance = tour_distance(candidate, distance_matrix)
            if candidate_distance < current_distance:
                return candidate, True, candidate_distance
    return tour.copy(), False, current_distance

In [ ]:

start_time = time.perf_counter()
candidate_tour, improved, new_distance = one_insertion_first_improvement(
    tour, current_distance, dist_matrix
)
elapsed = time.perf_counter() - start_time

print("Improved:", improved)
print("New distance:", new_distance)
print("Candidate tour:", candidate_tour)
print(f"Execution time: {elapsed * 1000:.3f} ms")

#### New method

In [ ]:
def one_insertion_first_improvement_test(
 tour: np.ndarray, current_distance: float, distance_matrix: np.ndarray
) -> tuple[np.ndarray, bool, float]:
    is_closed = tour[0] == tour[-1]
    length = len(tour) - 1 if is_closed else len(tour)
    if length < 3:
        return tour, False, current_distance

    def edge_cost(u: int, v: int) -> float:
        lower, upper = (u, v) if u <= v else (v, u)
        return distance_matrix[lower, upper]

    core = tour[:-1] if is_closed else tour

    for i in range(1, length):
        city = core[i]
        prev_i = core[i - 1] if i > 0 else (core[-1] if is_closed else None)
        next_i = core[(i + 1) % length] if (is_closed or i + 1 < length) else None

        removal_delta = 0.0
        if prev_i is not None:
            removal_delta -= edge_cost(prev_i, city)
        if next_i is not None:
            removal_delta -= edge_cost(city, next_i)
            if prev_i is not None:
                removal_delta += edge_cost(prev_i, next_i)

        core_removed = np.delete(core, i)
        reduced_len = len(core_removed)
        if reduced_len == 0:
            continue

        for j in range(1, length + 1):
            if i == j or j == i + 1:
                continue

            insert_idx = j
            if insert_idx > i:
                insert_idx -= 1

            if insert_idx < 0 or insert_idx > reduced_len:
                continue

            if is_closed:
                prev_new = core_removed[(insert_idx - 1) % reduced_len]
                next_new = core_removed[insert_idx % reduced_len]
            else:
                prev_new = core_removed[insert_idx - 1] if insert_idx > 0 else None
                next_new = core_removed[insert_idx] if insert_idx < reduced_len else None

            insertion_delta = 0.0
            if prev_new is not None and next_new is not None:
                insertion_delta -= edge_cost(prev_new, next_new)
            if prev_new is not None:
                insertion_delta += edge_cost(prev_new, city)
            if next_new is not None:
                insertion_delta += edge_cost(city, next_new)

            delta = removal_delta + insertion_delta
            if delta < 0:
                new_distance = current_distance + delta
                candidate = one_insertion(tour, i, j)
                return candidate, True, new_distance

    return tour, False, current_distance

In [ ]:
tour

In [ ]:
current_distance = tour_distance(tour, dist_matrix)
start_time = time.perf_counter()
candidate_tour, improved, new_distance = one_insertion_first_improvement_test(
    tour, current_distance, dist_matrix
)
elapsed = time.perf_counter() - start_time

print("Improved:", improved)
print("New distance:", new_distance)
print("Candidate tour:", candidate_tour)
print(f"Execution time: {elapsed * 1000:.3f} ms")

## Two exchange test

In [ ]:
def two_exchange(tour: np.ndarray, i: int, j: int) -> np.ndarray:
    """Swap two vertex positions in the tour (2-exchange move)."""
    is_closed = tour[0] == tour[-1]
    core = tour[:-1] if is_closed else tour
    new_core = core.copy()
    new_core[i], new_core[j] = new_core[j], new_core[i]
    if is_closed:
        return np.concatenate((new_core, [new_core[0]]))
    return new_core


In [ ]:
def two_exchange_first_improvement(
    tour: np.ndarray,
    current_distance: float,
    distance_matrix: np.ndarray,
) -> tuple[np.ndarray, bool, float]:
    """Brute-force 2-exchange that recomputes full tour distance each time."""
    length = len(tour) - 1 if tour[0] == tour[-1] else len(tour)
    if length < 3:
        return tour.copy(), False, current_distance

    for i in range(1, length - 1):
        for j in range(i + 1, length):
            candidate = two_exchange(tour, i, j)
            candidate_distance = tour_distance(candidate, distance_matrix)
            if candidate_distance < current_distance:
                return candidate, True, candidate_distance
    return tour.copy(), False, current_distance

In [ ]:
def two_exchange_first_improvement_delta(
    tour: np.ndarray,
    current_distance: float,
    distance_matrix: np.ndarray,
) -> tuple[np.ndarray, bool, float]:
    """First-improvement 2-exchange using incremental edge deltas."""
    is_closed = tour[0] == tour[-1]
    length = len(tour) - 1 if is_closed else len(tour)
    if length < 3:
        return tour.copy(), False, current_distance

    def edge_cost(u: int | None, v: int | None) -> float:
        if u is None or v is None:
            return 0.0
        lower, upper = (u, v) if u <= v else (v, u)
        return distance_matrix[lower, upper]

    core = tour[:-1] if is_closed else tour

    for i in range(1, length - 1):
        for j in range(i + 1, length):
            city_i = core[i]
            city_j = core[j]

            prev_i = core[i - 1] if i > 0 else (core[-1] if is_closed else None)
            next_i = core[(i + 1) % length] if (is_closed or i + 1 < length) else None

            prev_j = core[j - 1] if j > 0 else (core[-1] if is_closed else None)
            next_j = core[(j + 1) % length] if (is_closed or j + 1 < length) else None

            if j == i + 1:
                removed = edge_cost(prev_i, city_i) + edge_cost(city_j, next_j)
                added = edge_cost(prev_i, city_j) + edge_cost(city_i, next_j)
            else:
                removed = (
                    edge_cost(prev_i, city_i)
                    + edge_cost(city_i, next_i)
                    + edge_cost(prev_j, city_j)
                    + edge_cost(city_j, next_j)
                )
                added = (
                    edge_cost(prev_i, city_j)
                    + edge_cost(city_j, next_i)
                    + edge_cost(prev_j, city_i)
                    + edge_cost(city_i, next_j)
                )

            delta = added - removed
            if delta < 0:
                new_distance = current_distance + delta
                candidate = two_exchange(tour, i, j)
                return candidate, True, new_distance

    return tour.copy(), False, current_distance

In [ ]:
tour = np.array([0, 1, 2, 3, 4, 5, 0])
dist_matrix = np.array([
    [0.0, 10.0, 20.0, 25.0, 31.0, 40.0, 42.0],
    [10.0, 0.0, 15.0, 22.0, 27.0, 33.0, 35.0],
    [20.0, 15.0, 0.0, 18.0, 24.0, 30.0, 32.0],
    [25.0, 22.0, 18.0, 0.0, 16.0, 26.0, 28.0],
    [31.0, 27.0, 24.0, 16.0, 0.0, 21.0, 23.0],
    [40.0, 33.0, 30.0, 26.0, 21.0, 0.0, 19.0],
    [42.0, 35.0, 32.0, 28.0, 23.0, 19.0, 0.0],
])



In [ ]:
current_distance = tour_distance(tour, dist_matrix)

start = time.perf_counter()
candidate_baseline, improved_baseline, dist_baseline = two_exchange_first_improvement(
    tour, current_distance, dist_matrix
)
elapsed_baseline = time.perf_counter() - start

start = time.perf_counter()
candidate_delta, improved_delta, dist_delta = two_exchange_first_improvement_delta(
    tour, current_distance, dist_matrix
)
elapsed_delta = time.perf_counter() - start

print("Baseline improved:", improved_baseline, "distance:", dist_baseline)
print("Delta improved:", improved_delta, "distance:", dist_delta)
print("Tours equal:", np.array_equal(candidate_baseline, candidate_delta))
print(f"Baseline time: {elapsed_baseline * 1000:.3f} ms")
print(f"Delta time: {elapsed_delta * 1000:.3f} ms")

### Double bridge 

In [3]:
def double_bridge_move(
    tour: np.ndarray, a: int, b: int, c: int, d: int
) -> np.ndarray:
    """Apply a double-bridge move defined by four cut indices."""
    if len(tour) < 6:
        return tour.copy()

    is_closed = tour[0] == tour[-1]
    core = tour[:-1] if is_closed else tour
    n = len(core)
    if n < 6:
        return tour.copy()

    segment_1 = core[:a]
    segment_2 = core[a:b]
    segment_3 = core[b:c]
    segment_4 = core[c:d]
    segment_5 = core[d:]

    new_core = np.concatenate(
        (segment_1, segment_3, segment_2, segment_4, segment_5)
    )
    if is_closed:
        new_core = np.concatenate((new_core, [new_core[0]]))
    return new_core


In [9]:

def double_bridge_first_improvement(
     tour: np.ndarray, current_distance: float, distance_matrix
) -> tuple[np.ndarray, bool]:
    is_closed = tour[0] == tour[-1]
    length = len(tour) - 1 if is_closed else len(tour)
    if length < 6:
        return tour, False, current_distance

    # Exhaustive enumeration is O(n^4). Instead, sample a bounded number of candidate quadruples.
    max_checks = 250
    if max_checks == 0:
        return tour, False, current_distance

    indices = list(range(1, length))
    tried: set[tuple[int, int, int, int]] = set()
    checks = 0

    # We keep sampling unique quadruples until we reach the configured limit.
    while checks < max_checks and len(tried) < max_checks:
        print(f"Checks {checks}")
        a, b, c, d = sorted(random.sample(indices, 4))
        key = (a, b, c, d)
        if key in tried:
            continue
        tried.add(key)
        checks += 1

        candidate = double_bridge_move(tour, a, b, c, d)
        candidate_distance = tour_distance(candidate, distance_matrix)
        print(candidate_distance, current_distance)
        if candidate_distance < current_distance:
            return candidate, True, candidate_distance
    return tour, False, current_distance

In [ ]:
def double_bridge_first_improvement_delta(
   tour: np.ndarray, current_distance: float, distance_matrix
) -> tuple[np.ndarray, bool, float]:
    is_closed = tour[0] == tour[-1]
    length = len(tour) - 1 if is_closed else len(tour)
    if length < 6:
        return tour, False, current_distance

    def edge_cost(u: int | None, v: int | None) -> float:
        if u is None or v is None:
            return 0.0
        lower, upper = (u, v) if u <= v else (v, u)
        return distance_matrix[lower, upper]

    # Exhaustive enumeration is O(n^4). Instead, sample a bounded number of candidate quadruples.
    max_checks = 250
    if max_checks == 0:
        return tour, False, current_distance

    core = tour[:-1] if is_closed else tour
    indices = list(range(1, length))
    tried: set[tuple[int, int, int, int]] = set()
    checks = 0

    # We keep sampling unique quadruples until we reach the configured limit.
    while checks < max_checks and len(tried) < max_checks:
        a, b, c, d = sorted(random.sample(indices, 4))
        key = (a, b, c, d)
        if key in tried:
            continue
        tried.add(key)
        checks += 1

        prev_a = core[a - 1]
        head_s2 = core[a]
        tail_s2 = core[b - 1]
        head_s3 = core[b]
        tail_s3 = core[c - 1]
        head_s4 = core[c]

        removed = (
            edge_cost(prev_a, head_s2)
            + edge_cost(tail_s2, head_s3)
            + edge_cost(tail_s3, head_s4)
        )
        added = (
            edge_cost(prev_a, head_s3)
            + edge_cost(tail_s3, head_s2)
            + edge_cost(tail_s2, head_s4)
        )

        delta = added - removed
        if delta < 0:
            new_distance = current_distance + delta
            candidate = double_bridge_move(tour, a, b, c, d)
            return candidate, True, new_distance
    return tour, False, current_distance

In [8]:
current_distance = tour_distance(tour, dist_matrix)

start = time.perf_counter()
candidate_baseline, improved_baseline, dist_baseline = double_bridge_first_improvement(
    tour, current_distance, dist_matrix
)
elapsed_baseline = time.perf_counter() - start
print("Baseline improved:", improved_baseline, "distance:", dist_baseline)
print(f"Baseline time: {elapsed_baseline * 1000:.3f} ms")


135.0 120.0
147.0 120.0
134.0 120.0
134.0 120.0
148.0 120.0


KeyboardInterrupt: 

In [ ]:
start = time.perf_counter()

candidate_delta, improved_delta, dist_delta = double_bridge_first_improvement_delta(
    tour, current_distance, dist_matrix
)
elapsed_delta = time.perf_counter() - start

print("Delta improved:", improved_delta, "distance:", dist_delta)
print(f"Delta time: {elapsed_delta * 1000:.3f} ms")

In [ ]:
print("Tours equal:", np.array_equal(candidate_baseline, candidate_delta))